In [ ]:
# TASK1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder ,StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import warnings
warnings.filterwarnings('ignore')
print("All imported")

All imported


In [53]:
# TASK 2-TASK 3
# Basic data inspection and data cleaning and processing
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("AI-Based Hiring Prediction System.csv")

# Fix spaces IMMEDIATELY after loading ← this must come right after read_csv
df.columns = df.columns.str.strip()
import joblib
joblib.dump(df, 'model.pkl')
print("✅ Loaded & column names cleaned!")
print(df.columns.tolist())
print("First 5 rows:")
print(df.head())
print("Last 5 Rows:")
print(df.tail())
print("Random 5 rows: ")
print(df.sample(5,random_state=42))
print("Rows,Columns:",df.shape)
print("\n Columns:",df.columns.tolist())
print("\n Data types:\n",df.dtypes)
print("\nRecruiter Decision counts :\n",df['Recruiter Decision'].value_counts())
print("\nStatistics :\n",df.describe())


 # Drop columns we don't need
df.drop(columns=['Resume_ID', 'Name'], inplace=True)

# Drop AI Score — we're simulating independent prediction

df.drop(columns=['AI Score (0-100)'], inplace=True)
# Convert target: Hire → 1, Reject → 0
df['Recruiter Decision'] = df['Recruiter Decision'].map({'Hire': 1, 'Reject': 0})

# Check for missing values
print("Missing values per column:\n", df.isnull().sum())

# If there are missing values in numeric cols → fill with median
# df['Experience (Years)'].fillna(df['Experience (Years)'].median(), inplace=True)
# For text cols → fill with empty string
# df['Skills'].fillna('', inplace=True)
#  Fix the 274 missing Certifications — fill with 'None'
df['Certifications'] = df['Certifications'].fillna('None')

print("Missing values after fix:\n", df.isnull().sum())
print("\nShape:", df.shape)
print("\nFirst row:\n", df.iloc[0])
# Confirm data types look right
print("\nData types after cleaning:\n", df.dtypes)
print("\nTarget value counts:\n", df['Recruiter Decision'].value_counts())
import os
size = os.path.getsize('model.pkl') / (1024 * 1024)
print(f"Model size: {size:.2f} MB")

Saving AI-Based Hiring Prediction System.csv to AI-Based Hiring Prediction System (9).csv
✅ Loaded & column names cleaned!
['Resume_ID', 'Name', 'Skills', 'Experience (Years)', 'Education', 'Certifications', 'Job Role', 'Recruiter Decision', 'Salary Expectation ($)', 'Projects Count', 'AI Score (0-100)']
First 5 rows:
   Resume_ID              Name                                        Skills  \
0          1        Ashley Ali                      TensorFlow, NLP, Pytorch   
1          2      Wesley Roman  Deep Learning, Machine Learning, Python, SQL   
2          3     Corey Sanchez         Ethical Hacking, Cybersecurity, Linux   
3          4  Elizabeth Carney                   Python, Pytorch, TensorFlow   
4          5        Julie Hill                              SQL, React, Java   

   Experience (Years) Education                Certifications  \
0                  10      B.Sc                           NaN   
1                  10       MBA                     Google ML   
2   

The dataset contains structured tabuar data candidates applying for different technical roles
It includes: Personal identifiers(Resume_id,Name)
Candidate attributes(Skills,Experience ,Education,Certifications,ProjectsCount,Salary,Expectation,AIScore)
job-related info (job Role,Recruiter Decision)
This is essentially a classification dataset for hiring predictions,where features describe the candidates profile.
TARGET VARIABLE:values are either higher or reject


In [ ]:
# TASK4 -TASK 5 TASK 6 -TASK 7
import re
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("AI-Based Hiring Prediction System.csv")

df['combined_text']=(
                     df['Skills'] + ' ' +
                     df['Certifications']+' '+
                     df['Job Role'])
def clean_text(text):
  text=str(text).lower()
  text = re.sub(r'[^a-z0-9\s]', ' ', text)  # remove special chars like commas, dots
  text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
  return text
df['combined_text']= df['combined_text'].apply(clean_text)
print(df['combined_text'].head(3))
# TASK5
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=28)
text_features=tfidf.fit_transform(df['combined_text'])
print("TF-IDF vocabulary sample:",list(tfidf.vocabulary_.keys())[:20])
print("Shape of text features:",text_features.shape)
#  TASK 6
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['Education_encoded']=le.fit_transform(df['Education'])
print("Education mapping:")
for i , cls in enumerate(le.classes_):
  print(f"(cls)+(i)")
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler




numeric_features=df[['Experience (Years)','Salary Expectation ($)','Projects Count']].values
education_feature=df['Education_encoded'].values.reshape(-1,1).astype(np.float32)
y=df['Recruiter Decision'].values
scaler=StandardScaler()
numeric_scaled=scaler.fit_transform(numeric_features)
X=sp.hstack([numeric_scaled,
             education_feature,
             text_features])
print("Final X shape:",X.shape)
print("y shape: ",y.shape)
print("Class distribution-Hire: ",y.sum(),"|Reject:",(y==0).sum())
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("\nTraining samples:",X_train.shape[0])
print("Testing samples:",X_test.shape[0])

Saving AI-Based Hiring Prediction System.csv to AI-Based Hiring Prediction System (7).csv
0                                                  nan
1    deep learning machine learning python sql goog...
2    ethical hacking cybersecurity linux deep learn...
Name: combined_text, dtype: object
TF-IDF vocabulary sample: ['nan', 'deep', 'learning', 'machine', 'python', 'sql', 'google', 'ml', 'data', 'scientist', 'ethical', 'hacking', 'cybersecurity', 'linux', 'specialization', 'analyst', 'pytorch', 'tensorflow', 'aws', 'certified']
Shape of text features: (1000, 28)
Education mapping:
(cls)+(i)
(cls)+(i)
(cls)+(i)
(cls)+(i)
(cls)+(i)
Final X shape: (1000, 32)
y shape:  (1000,)
Class distribution-Hire:  HireHireHireHireHireHireHireHireHireRejectHireRejectHireHireRejectRejectHireHireHireHireHireHireHireHireHireHireRejectRejectHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireHireRejectHireRejectRejectHireRejectRejectHireHireHireHireHireHireRejectHireHir

In [ ]:
# ============================================================
# MASTER CELL — Run this once, top to bottom
# ============================================================
import pandas as pd
import numpy as np
import re
import scipy.sparse as sp
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# ---- STEP 1: Load ----
df = pd.read_csv('AI-Based Hiring Prediction System.csv')
print("Loaded:", df.shape)

# ---- STEP 2: Drop useless columns ----
df.drop(columns=['Resume_ID', 'Name', 'AI Score (0-100)'], inplace=True)

# ---- STEP 3: Fix target FIRST — before anything else ----
df['Recruiter Decision'] = df['Recruiter Decision'].map({'Hire': 1, 'Reject': 0})
print("Target fixed. Hire:", df['Recruiter Decision'].sum(),
      "| Reject:", (df['Recruiter Decision'] == 0).sum())

# ---- STEP 4: Fill ALL missing values ----
df['Skills'] = df['Skills'].fillna('')
df['Certifications'] = df['Certifications'].fillna('None')
df['Job Role'] = df['Job Role'].fillna('')
print("Missing values left:", df.isnull().sum().sum())  # should be 0

# ---- STEP 5: Build combined_text ----
df['combined_text'] = df['Skills'] + ' ' + df['Certifications'] + ' ' + df['Job Role']

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['combined_text'] = df['combined_text'].apply(clean_text)
print("Sample text row 0:", df['combined_text'].iloc[0])  # should NOT be nan

# ---- STEP 6: TF-IDF ----
tfidf = TfidfVectorizer(max_features=100)
text_features = tfidf.fit_transform(df['combined_text'])
print("TF-IDF shape:", text_features.shape)  # should be (1000, 100)

# ---- STEP 7: Encode Education ----
le = LabelEncoder()
df['Education_encoded'] = le.fit_transform(df['Education'])
print("Education classes:", list(le.classes_))

# ---- STEP 8: Separate features and target ----
numeric_features = df[['Experience (Years)',
                        'Salary Expectation ($)',
                        'Projects Count']].values.astype(np.float32)
education_feature = df['Education_encoded'].values.reshape(-1, 1).astype(np.float32)
y = df['Recruiter Decision'].values.astype(np.int32)

print("y unique values:", np.unique(y))  # must be [0, 1] not strings

# ---- STEP 9: Scale numeric features ----
scaler = StandardScaler()
numeric_scaled = scaler.fit_transform(numeric_features)

# ---- STEP 10: Combine everything into X ----
X = sp.hstack([
    sp.csr_matrix(numeric_scaled),
    sp.csr_matrix(education_feature),
    text_features
])
print("Final X shape:", X.shape)  # should be (1000, 104)

# ---- STEP 11: Train-test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("\n✅ All preprocessing done! Ready for model training.")

Loaded: (1000, 11)
Target fixed. Hire: 812 | Reject: 188
Missing values left: 0
Sample text row 0: tensorflow nlp pytorch none ai researcher
TF-IDF shape: (1000, 28)
Education classes: ['B.Sc', 'B.Tech', 'M.Tech', 'MBA', 'PhD']
y unique values: [0 1]
Final X shape: (1000, 32)
Train: 800 | Test: 200

✅ All preprocessing done! Ready for model training.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,classification_report
# ------LOGISTIC REGRESSION-----MODEL1
lr=LogisticRegression(max_iter=1000)
lr.fit(X_train,y_train)
lr_pred=lr.predict(X_test)
# ------RANDOM FOREST-MODEL 2------
rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train,y_train)
rf_pred=rf.predict(X_test)
# ------SVM-MODEL3------
svm=SVC(kernel='rbf',probability=True)
svm.fit(X_train,y_train)
svm_pred=svm.predict(X_test)
# ----KNN-MODEL 4-----
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train,y_train)
knn_pred=knn.predict(X_test)
print("All 4 models trained")





All 4 models trained


In [ ]:
models={'Logistic Regression': lr_pred,
        'Random Forest': rf_pred,
        'SVM':            svm_pred,
        'KNN':             knn_pred

        }
print("=" * 45)
print(f"{'MOdel': <25}{'Accuracy':>10}")
print("=" * 45)
for name,pred in models.items():
  acc=accuracy_score(y_test,pred)
  print(f"{name:<25}{acc*100:>9.2f}%")
print("=" * 45)
print("\n--- Logistic Regression Report ---")
print(classification_report(y_test, lr_pred, target_names=['Reject','Hire']))

print("\n--- Random Forest Report ---")
print(classification_report(y_test, rf_pred, target_names=['Reject','Hire']))

print("\n--- SVM Report ---")
print(classification_report(y_test, svm_pred, target_names=['Reject','Hire']))

print("\n--- KNN Report ---")
print(classification_report(y_test, knn_pred, target_names=['Reject','Hire']))


MOdel                      Accuracy
Logistic Regression          98.00%
Random Forest                95.50%
SVM                          97.00%
KNN                          96.00%

--- Logistic Regression Report ---
              precision    recall  f1-score   support

      Reject       0.99      0.98      0.99       154
        Hire       0.94      0.98      0.96        46

    accuracy                           0.98       200
   macro avg       0.97      0.98      0.97       200
weighted avg       0.98      0.98      0.98       200


--- Random Forest Report ---
              precision    recall  f1-score   support

      Reject       0.94      1.00      0.97       154
        Hire       1.00      0.80      0.89        46

    accuracy                           0.95       200
   macro avg       0.97      0.90      0.93       200
weighted avg       0.96      0.95      0.95       200


--- SVM Report ---
              precision    recall  f1-score   support

      Reject       0.97  

In [ ]:
# TASK 12
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
lr_pipeline=Pipeline([('classifier',LogisticRegression(max_iter=1000))])

# Parameters to try
lr_params = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(
    lr_pipeline,
    lr_params,
    cv=5,              # 5-fold cross validation
    scoring='f1',      # optimize for F1, not just accuracy
    n_jobs=-1          # use all CPU cores
)

lr_grid.fit(X_train, y_train)

print("=== Logistic Regression GridSearch ===")
print("Best C value:", lr_grid.best_params_)
print("Best CV F1 score:", round(lr_grid.best_score_ * 100, 2), "%")

# ============================================
# Pipeline 2: SVM + GridSearch
# ============================================
svm_pipeline = Pipeline([
    ('classifier', SVC(probability=True))
])

svm_params = {
    'classifier__C':      [0.1, 1, 10],
    'classifier__kernel': ['linear', 'rbf']
}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print("\n=== SVM GridSearch ===")
print("Best params:", svm_grid.best_params_)
print("Best CV F1 score:", round(svm_grid.best_score_ * 100, 2), "%")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan]
  warnings.warn(


=== Logistic Regression GridSearch ===
Best C value: {'classifier__C': 0.01}
Best CV F1 score: nan %

=== SVM GridSearch ===
Best params: {'classifier__C': 10, 'classifier__kernel': 'linear'}
Best CV F1 score: 98.25 %


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Pick whichever had better CV score
best_model = svm_grid  # change to svm_grid if SVM won

# Predict on test set
best_pred = best_model.predict(X_test)

print("=== Final Best Model Evaluation ===")
print("Test Accuracy:", round(accuracy_score(y_test, best_pred) * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['Reject', 'Hire']))
print("Best Parameters:", best_model.best_params_)

=== Final Best Model Evaluation ===
Test Accuracy: 97.5 %

Classification Report:
              precision    recall  f1-score   support

      Reject       0.99      0.98      0.98       162
        Hire       0.92      0.95      0.94        38

    accuracy                           0.97       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.98      0.97      0.98       200

Best Parameters: {'classifier__C': 10, 'classifier__kernel': 'linear'}


In [ ]:
def predict_candidate(skills, experience, education, certifications, projects_count, salary_expectation):
    raw_text = skills + ' ' + certifications + ' ' + education
    clean = clean_text(raw_text)
    text_vec = tfidf.transform([clean])
    edu_lower = education.strip()
    if edu_lower not in le.classes_:
        print(f"Unknown education. Valid: {list(le.classes_)}")
        return None
    edu_encoded = le.transform([edu_lower])[0]
    edu_feature = np.array([[edu_encoded]], dtype=np.float32)
    numeric = np.array([[experience, salary_expectation, projects_count]], dtype=np.float32)
    numeric_scaled = scaler.transform(numeric)
    X_input = sp.hstack([sp.csr_matrix(numeric_scaled), sp.csr_matrix(edu_feature), text_vec])

    prediction = svm_grid.predict(X_input)[0]
    probability = svm_grid.predict_proba(X_input)[0]

    # classes are 'Hire' and 'Reject' strings — find correct index
    classes = list(svm_grid.best_estimator_.named_steps['classifier'].classes_)
    hire_index   = classes.index('Hire')
    reject_index = classes.index('Reject')

    hire_prob   = round(probability[hire_index] * 100, 2)
    reject_prob = round(probability[reject_index] * 100, 2)
    result = "✅ HIRED" if prediction == 'Hire' else "❌ REJECTED"

    print("=" * 40)
    print("      AI HIRING PREDICTION SYSTEM")
    print("=" * 40)
    print(f"Skills         : {skills}")
    print(f"Experience     : {experience} years")
    print(f"Education      : {education}")
    print(f"Certifications : {certifications}")
    print(f"Projects       : {projects_count}")
    print(f"Salary Exp.    : ${salary_expectation}")
    print("-" * 40)
    print(f"PREDICTION     : {result}")
    print(f"Hire Prob.     : {hire_prob}%")
    print(f"Reject Prob.   : {reject_prob}%")
    print("=" * 40)

    return prediction, hire_prob

print("✅ Function defined!")

✅ Function defined!


In [ ]:
# Strong candidate
predict_candidate("Python Machine Learning TensorFlow", 5, "B.Tech", "AWS Certified", 8, 90000)
print()
# Weak candidate
predict_candidate("Java Networking", 0, "B.Sc", "None", 0, 115000)
print()
# Mid candidate
predict_candidate("Python React Machine Learning", 1, "B.Tech", "None", 4, 60000)

      AI HIRING PREDICTION SYSTEM
Skills         : Python Machine Learning TensorFlow
Experience     : 5 years
Education      : B.Tech
Certifications : AWS Certified
Projects       : 8
Salary Exp.    : $90000
----------------------------------------
PREDICTION     : ✅ HIRED
Hire Prob.     : 100.0%
Reject Prob.   : 0.0%

      AI HIRING PREDICTION SYSTEM
Skills         : Java Networking
Experience     : 0 years
Education      : B.Sc
Certifications : None
Projects       : 0
Salary Exp.    : $115000
----------------------------------------
PREDICTION     : ❌ REJECTED
Hire Prob.     : 0.0%
Reject Prob.   : 100.0%

      AI HIRING PREDICTION SYSTEM
Skills         : Python React Machine Learning
Experience     : 1 years
Education      : B.Tech
Certifications : None
Projects       : 4
Salary Exp.    : $60000
----------------------------------------
PREDICTION     : ✅ HIRED
Hire Prob.     : 87.18%
Reject Prob.   : 12.82%


('Hire', np.float64(87.18))

In [ ]:
# TASK 14
print("""
╔══════════════════════════════════════════════════════╗
║         AI HIRING PREDICTION SYSTEM — CONCLUSION     ║
╚══════════════════════════════════════════════════════╝

DATASET UNDERSTANDING:
- 1000 synthetic resumes with mixed text + numeric data
- Target: Recruiter Decision (81% Hire, 19% Reject)
- Key challenge: class imbalance + raw text features

KEY PREPROCESSING STEPS:
- Dropped Resume_ID, Name, AI Score (non-predictive)
- Filled 274 missing Certifications with 'None'
- Combined Skills + Certifications + Job Role → TF-IDF (28 features)
- Label encoded Education (5 categories)
- StandardScaler on Experience, Salary, Projects Count
- Final feature matrix: 1000 × 32

MODEL PERFORMANCE SUMMARY:
┌─────────────────────┬──────────┐
│ Model               │ Accuracy │
├─────────────────────┼──────────┤
│ Logistic Regression │  98.00%  │
│ SVM (tuned)         │  97.50%  │
│ KNN                 │  96.00%  │
│ Random Forest       │  95.50%  │
└─────────────────────┴──────────┘

BEST MODEL: Logistic Regression (manual) / SVM (after GridSearch)
- GridSearch found C=10, kernel=linear as optimal SVM params
- F1-weighted score: 98.25% via cross validation

WHAT I LEARNED:
- Data cleaning affects model quality more than model choice
- F1 score matters more than accuracy on imbalanced datasets
- GridSearchCV finds better parameters than manual guessing
- Pipelines make production deployment clean and reliable
- Always use stratify=y in train_test_split for imbalanced data

REAL WORLD CONNECTION:
This project simulates actual AI resume screening tools used
by companies like LinkedIn, Naukri, and HireVue. In production,
this model would auto-filter thousands of resumes in seconds,
shortlisting only the strongest candidates for human review —
saving HR teams hours of manual screening work.
""")


╔══════════════════════════════════════════════════════╗
║         AI HIRING PREDICTION SYSTEM — CONCLUSION     ║
╚══════════════════════════════════════════════════════╝

DATASET UNDERSTANDING:
- 1000 synthetic resumes with mixed text + numeric data
- Target: Recruiter Decision (81% Hire, 19% Reject)
- Key challenge: class imbalance + raw text features

KEY PREPROCESSING STEPS:
- Dropped Resume_ID, Name, AI Score (non-predictive)
- Filled 274 missing Certifications with 'None'
- Combined Skills + Certifications + Job Role → TF-IDF (28 features)
- Label encoded Education (5 categories)
- StandardScaler on Experience, Salary, Projects Count
- Final feature matrix: 1000 × 32

MODEL PERFORMANCE SUMMARY:
┌─────────────────────┬──────────┐
│ Model               │ Accuracy │
├─────────────────────┼──────────┤
│ Logistic Regression │  98.00%  │
│ SVM (tuned)         │  97.50%  │
│ KNN                 │  96.00%  │
│ Random Forest       │  95.50%  │
└─────────────────────┴──────────┘

BEST MOD